# ClipCap caption evaluation

Notebook này chỉ thực hiện **đánh giá metric**. Input là `predictions.jsonl` và `run_config.json` do `clipcap_caption_generation.ipynb` sinh ra. Notebook không chạy inference và không thay đổi caption.

Metric: CIDEr là tiêu chí chính, BLEU-4 là tiêu chí phụ, CLIPScore chỉ là tín hiệu semantic bổ sung vì cùng CLIP model đã tham gia reranking.

## 1. Cấu hình evaluation

`SPLIT_NAME`, `RUN_TAG`, `SEED` và danh sách subset phải trùng với inference run. Test bị khóa mặc định. Notebook Colab end-to-end truyền cấu hình qua biến môi trường.

In [ ]:
from __future__ import annotations

import csv
import hashlib
import importlib.util
import json
import os
import re
import shutil
import sys
from datetime import datetime, timezone
from pathlib import Path


def find_project_root(start: Path) -> Path:
    resolved = start.expanduser().resolve()
    for candidate in (resolved, *resolved.parents):
        if (candidate / 'src' / 'config' / 'common_config.py').is_file():
            return candidate
    raise FileNotFoundError(
        'Không tìm thấy project root. Đặt ZFS_CLIP_PROJECT_ROOT hoặc mở notebook '
        'từ repository.'
    )


def env_bool(name: str, default: bool) -> bool:
    value = os.environ.get(name)
    if value is None:
        return default
    normalized = value.strip().lower()
    if normalized in {'1', 'true', 'yes', 'on'}:
        return True
    if normalized in {'0', 'false', 'no', 'off'}:
        return False
    raise ValueError(f'{name} phải là biến boolean')


def env_path(name: str, default: Path | None = None) -> Path | None:
    value = os.environ.get(name)
    return Path(value).expanduser() if value else default


project_override = env_path('ZFS_CLIP_PROJECT_ROOT')
PROJECT_ROOT = find_project_root(project_override or Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config.clipcap_config import (
    CLIPCAP_OUTPUT_ROOT,
    CLIPCAP_TRAIN_SUBSETS,
    CLIP_MODEL_NAME,
)
from src.config.common_config import SPLIT_DIR

SPLIT_NAME = os.environ.get('ZFS_CLIP_SPLIT_NAME', 'val')
ALLOW_TEST_EVALUATION = env_bool('ZFS_CLIP_ALLOW_TEST', False)
RUN_TAG = os.environ.get('ZFS_CLIP_RUN_TAG', 'baseline_v1')
SEED = int(os.environ.get('ZFS_CLIP_SEED', '42'))
subset_text = os.environ.get(
    'ZFS_CLIP_SUBSETS',
    ','.join(CLIPCAP_TRAIN_SUBSETS),
)
SUBSETS_TO_EVALUATE = tuple(
    item.strip() for item in subset_text.split(',') if item.strip()
)
EXPECTED_REFERENCES_PER_IMAGE = int(
    os.environ.get('ZFS_CLIP_REFERENCES_PER_IMAGE', '5')
)
DEVICE = os.environ.get('ZFS_CLIP_DEVICE', 'auto')
CLIP_BATCH_SIZE = int(os.environ.get('ZFS_CLIP_METRIC_BATCH_SIZE', '32'))

REFERENCES_PATH = env_path(
    'ZFS_CLIP_MANIFEST_PATH',
    SPLIT_DIR / f'{SPLIT_NAME}.json',
)
INFERENCE_OUTPUT_BASE = env_path(
    'ZFS_CLIP_INFERENCE_OUTPUT_BASE',
    CLIPCAP_OUTPUT_ROOT / 'evaluation',
)
INFERENCE_OUTPUT_ROOT = INFERENCE_OUTPUT_BASE / SPLIT_NAME / RUN_TAG
METRICS_OUTPUT_BASE = env_path(
    'ZFS_CLIP_METRICS_OUTPUT_BASE',
    CLIPCAP_OUTPUT_ROOT / 'metrics',
)
METRICS_OUTPUT_DIR = (
    METRICS_OUTPUT_BASE / SPLIT_NAME / RUN_TAG / f'seed_{SEED}'
)

if SPLIT_NAME not in {'val', 'test'}:
    raise ValueError("SPLIT_NAME phải là 'val' hoặc 'test'")
if SPLIT_NAME == 'test' and not ALLOW_TEST_EVALUATION:
    raise RuntimeError(
        'Test evaluation đang bị khóa. Chỉ bật sau khi đã chốt validation.'
    )
if not RUN_TAG or not re.fullmatch(r'[A-Za-z0-9._-]+', RUN_TAG):
    raise ValueError('RUN_TAG chứa ký tự không hợp lệ')
if not SUBSETS_TO_EVALUATE:
    raise ValueError('SUBSETS_TO_EVALUATE không được rỗng')
unknown = set(SUBSETS_TO_EVALUATE) - set(CLIPCAP_TRAIN_SUBSETS)
if unknown:
    raise ValueError(f'Unknown subsets: {sorted(unknown)}')

required_packages = {'pycocoevalcap': 'pycocoevalcap', 'transformers': 'transformers'}
missing_packages = [
    package for package, module in required_packages.items()
    if importlib.util.find_spec(module) is None
]
if missing_packages:
    raise RuntimeError(
        f'Thiếu package {missing_packages}. Chạy notebook Colab setup trước.'
    )
if shutil.which('java') is None:
    raise RuntimeError('CIDEr/BLEU-4 chuẩn COCO cần Java trong PATH')

print(f'Input inference run: {INFERENCE_OUTPUT_ROOT}')
print(f'Metrics output: {METRICS_OUTPUT_DIR}')
print(f'Subsets: {list(SUBSETS_TO_EVALUATE)}')

## 2. Đọc và xác minh prediction

Notebook từ chối prediction thiếu ảnh, dư ảnh, trùng ID hoặc không khớp `run_config.json`. Ground-truth chỉ được dùng trong metric, không tham gia inference hoặc reranking.

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def reject_duplicate_keys(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f'Duplicate JSON key: {key!r}')
        result[key] = value
    return result


def load_json_strict(path: Path):
    with path.open('r', encoding='utf-8') as file:
        return json.load(file, object_pairs_hook=reject_duplicate_keys)


def load_references(path: Path, expected_per_image: int) -> dict[str, list[str]]:
    if not path.is_file():
        raise FileNotFoundError(f'Không tìm thấy reference manifest: {path}')
    payload = load_json_strict(path)
    if not isinstance(payload, dict) or not payload:
        raise TypeError('Reference manifest phải là JSON object không rỗng')
    references = {}
    for image_id, captions in payload.items():
        if not isinstance(image_id, str) or not image_id.strip():
            raise ValueError('image_id phải là chuỗi không rỗng')
        if not isinstance(captions, list) or len(captions) != expected_per_image:
            raise ValueError(
                f'{image_id} không có đúng {expected_per_image} references'
            )
        cleaned = []
        for caption in captions:
            if not isinstance(caption, str) or not caption.strip():
                raise ValueError(f'{image_id} chứa reference không hợp lệ')
            cleaned.append(caption.strip())
        references[image_id] = cleaned
    return references


def load_predictions(path: Path) -> dict[str, str]:
    if not path.is_file():
        raise FileNotFoundError(f'Không tìm thấy prediction: {path}')
    predictions = {}
    with path.open('r', encoding='utf-8') as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue
            try:
                record = json.loads(line, object_pairs_hook=reject_duplicate_keys)
            except json.JSONDecodeError as error:
                raise ValueError(f'JSONL lỗi tại {path}:{line_number}') from error
            image_id = record.get('image_id')
            caption = record.get('caption')
            if not isinstance(image_id, str) or not image_id.strip():
                raise ValueError(f'image_id lỗi tại {path}:{line_number}')
            if not isinstance(caption, str) or not caption.strip():
                raise ValueError(f'caption lỗi tại {path}:{line_number}')
            if image_id in predictions:
                raise ValueError(f'Duplicate prediction: {image_id}')
            predictions[image_id] = caption.strip()
    if not predictions:
        raise ValueError(f'Prediction file rỗng: {path}')
    return predictions


references = load_references(REFERENCES_PATH, EXPECTED_REFERENCES_PER_IMAGE)
run_config_path = INFERENCE_OUTPUT_ROOT / 'run_config.json'
if not run_config_path.is_file():
    raise FileNotFoundError(f'Không tìm thấy run config: {run_config_path}')
run_config = load_json_strict(run_config_path)

required_run_values = {
    'manifest_sha256': sha256_file(REFERENCES_PATH),
    'seed': SEED,
    'checkpoint': 'final.pt',
    'training_policy': 'fixed_epoch',
    'prompt': None,
    'num_images': len(references),
    'clip_model': CLIP_MODEL_NAME,
}
for key, expected_value in required_run_values.items():
    if run_config.get(key) != expected_value:
        raise ValueError(
            f'run_config mismatch tại {key}: '
            f'{run_config.get(key)!r} != {expected_value!r}'
        )
if not set(SUBSETS_TO_EVALUATE).issubset(set(run_config.get('subsets', []))):
    raise ValueError('Subset cần đánh giá không nằm trong inference run')

feature_cache_value = run_config.get('feature_cache')
if not feature_cache_value:
    raise ValueError('run_config không chứa feature_cache cho CLIPScore')
FEATURE_CACHE_PATH = Path(feature_cache_value)
if not FEATURE_CACHE_PATH.is_file():
    raise FileNotFoundError(f'Không tìm thấy feature cache: {FEATURE_CACHE_PATH}')

experiment_specs = {}
predictions_by_experiment = {}
coverage_by_experiment = {}
reference_ids = set(references)
for subset_name in SUBSETS_TO_EVALUATE:
    label = f'{subset_name}/seed_{SEED}'
    prediction_path = (
        INFERENCE_OUTPUT_ROOT / subset_name / f'seed_{SEED}' / 'predictions.jsonl'
    )
    predictions = load_predictions(prediction_path)
    missing = reference_ids - set(predictions)
    extra = set(predictions) - reference_ids
    if missing or extra:
        raise ValueError(
            f'{label}: missing={len(missing)}, extra={len(extra)}'
        )
    experiment_specs[label] = {
        'subset_name': subset_name,
        'seed': SEED,
        'predictions_path': prediction_path,
    }
    predictions_by_experiment[label] = predictions
    coverage_by_experiment[label] = {
        'num_images': len(references),
        'num_predictions': len(predictions),
        'num_reference_captions': sum(len(items) for items in references.values()),
    }
    print(f'{label}: {coverage_by_experiment[label]}')

## 3. CIDEr và BLEU-4 chuẩn COCO

Caption prediction và năm reference được tokenize bằng PTBTokenizer trước khi tính metric. Giá trị được nhân 100 khi báo cáo để dễ đọc; CIDEr sau khi nhân 100 không phải phần trăm.

In [ ]:
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer


def evaluate_coco_metrics(references, predictions):
    image_ids = sorted(references)
    raw_gts = {
        image_id: [{'caption': caption} for caption in references[image_id]]
        for image_id in image_ids
    }
    raw_res = {
        image_id: [{'caption': predictions[image_id]}]
        for image_id in image_ids
    }
    tokenizer = PTBTokenizer(verbose=False)
    gts = tokenizer.tokenize(raw_gts)
    res = tokenizer.tokenize(raw_res)
    bleu_score, bleu_per_image = Bleu(4).compute_score(gts, res, verbose=0)
    cider_score, cider_per_image = Cider().compute_score(gts, res)
    per_image = {
        image_id: {
            'CIDEr': float(cider_value) * 100.0,
            'BLEU-4': float(bleu_value) * 100.0,
        }
        for image_id, cider_value, bleu_value in zip(
            image_ids, cider_per_image, bleu_per_image[3]
        )
    }
    return {
        'CIDEr': float(cider_score) * 100.0,
        'BLEU-4': float(bleu_score[3]) * 100.0,
        'per_image': per_image,
    }


coco_results = {}
for label, predictions in predictions_by_experiment.items():
    print(f'Computing CIDEr and BLEU-4 for {label}...')
    coco_results[label] = evaluate_coco_metrics(references, predictions)
    print(
        f"  CIDEr={coco_results[label]['CIDEr']:.4f} | "
        f"BLEU-4={coco_results[label]['BLEU-4']:.4f}"
    )

## 4. CLIPScore từ feature cache

Image feature được tái sử dụng trực tiếp từ cache của inference; notebook chỉ mã hóa caption. Điều này tránh đọc và mã hóa lại toàn bộ ảnh trên Colab.

In [ ]:
import torch
import torch.nn.functional as functional
from transformers import CLIPModel, CLIPProcessor

from src.clipcap.inference.features import load_feature_cache


def unwrap_features(value):
    return value.pooler_output if hasattr(value, 'pooler_output') else value


def evaluate_clipscore_from_cache(
    references,
    predictions_by_experiment,
    feature_cache_path: Path,
    model_name: str,
    batch_size: int,
    device_name: str,
):
    if device_name == 'auto':
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    else:
        device = torch.device(device_name)
    if device.type == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('CUDA được yêu cầu nhưng không khả dụng')

    image_ids = sorted(references)
    cached = load_feature_cache(feature_cache_path, image_ids)
    image_features = torch.stack([cached[image_id] for image_id in image_ids]).float()
    image_features = functional.normalize(image_features, dim=-1)

    print(f'Loading CLIP text encoder {model_name} on {device}...')
    processor = CLIPProcessor.from_pretrained(model_name)
    model = CLIPModel.from_pretrained(model_name).to(device).eval()
    for parameter in model.parameters():
        parameter.requires_grad = False

    results = {}
    with torch.inference_mode():
        for label, predictions in predictions_by_experiment.items():
            per_image_scores = {}
            for start in range(0, len(image_ids), batch_size):
                batch_ids = image_ids[start:start + batch_size]
                captions = [predictions[image_id] for image_id in batch_ids]
                text_inputs = processor(
                    text=captions,
                    padding=True,
                    truncation=True,
                    return_tensors='pt',
                )
                text_features = unwrap_features(
                    model.get_text_features(
                        input_ids=text_inputs['input_ids'].to(device),
                        attention_mask=text_inputs['attention_mask'].to(device),
                    )
                )
                text_features = functional.normalize(text_features.float(), dim=-1)
                batch_image_features = image_features[
                    start:start + len(batch_ids)
                ].to(device)
                scores = (
                    100.0 * (batch_image_features * text_features).sum(dim=-1)
                ).clamp(min=0.0)
                for image_id, score in zip(batch_ids, scores.cpu().tolist()):
                    per_image_scores[image_id] = float(score)
            results[label] = {
                'CLIPScore': sum(per_image_scores.values()) / len(per_image_scores),
                'per_image': per_image_scores,
            }

    del model
    del processor
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return results


print(
    'Warning: CLIPScore dùng cùng CLIP model đã rerank caption; '
    'không dùng làm tiêu chí tuning chính.'
)
clip_results = evaluate_clipscore_from_cache(
    references,
    predictions_by_experiment,
    FEATURE_CACHE_PATH,
    CLIP_MODEL_NAME,
    CLIP_BATCH_SIZE,
    DEVICE,
)
for label, result in clip_results.items():
    print(f"{label}: CLIPScore={result['CLIPScore']:.4f}")

## 5. Lưu và in kết quả

`summary.json` phục vụ so sánh cấu hình; `per_image_scores.csv` dùng để phân tích các trường hợp tốt/xấu. Kết quả được xếp theo CIDEr rồi BLEU-4.

In [ ]:
summary_rows = []
per_image_rows = []
for label, spec in experiment_specs.items():
    summary_rows.append({
        'experiment': label,
        'subset_name': spec['subset_name'],
        'seed': spec['seed'],
        'CIDEr': coco_results[label]['CIDEr'],
        'BLEU-4': coco_results[label]['BLEU-4'],
        'CLIPScore': clip_results[label]['CLIPScore'],
    })
    for image_id in sorted(references):
        per_image_rows.append({
            'experiment': label,
            'image_id': image_id,
            'prediction': predictions_by_experiment[label][image_id],
            'CIDEr': coco_results[label]['per_image'][image_id]['CIDEr'],
            'BLEU-4': coco_results[label]['per_image'][image_id]['BLEU-4'],
            'CLIPScore': clip_results[label]['per_image'][image_id],
        })

summary_rows.sort(key=lambda row: (-row['CIDEr'], -row['BLEU-4']))
for rank, row in enumerate(summary_rows, start=1):
    row['rank'] = rank

METRICS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
summary_path = METRICS_OUTPUT_DIR / 'summary.json'
per_image_path = METRICS_OUTPUT_DIR / 'per_image_scores.csv'
artifact = {
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'split': SPLIT_NAME,
    'run_tag': RUN_TAG,
    'ranking_policy': ['CIDEr', 'BLEU-4'],
    'supplementary_metric': 'CLIPScore',
    'metric_scales': {
        'CIDEr': 'raw CIDEr multiplied by 100; not a percentage',
        'BLEU-4': 'corpus BLEU-4 multiplied by 100',
        'CLIPScore': 'max(100 * cosine_similarity, 0)',
    },
    'references_path': str(REFERENCES_PATH.resolve()),
    'references_sha256': sha256_file(REFERENCES_PATH),
    'feature_cache_path': str(FEATURE_CACHE_PATH.resolve()),
    'run_config': run_config,
    'run_config_path': str(run_config_path.resolve()),
    'run_config_sha256': sha256_file(run_config_path),
    'coverage': coverage_by_experiment,
    'results': summary_rows,
    'prediction_files': {
        label: {
            'path': str(spec['predictions_path'].resolve()),
            'sha256': sha256_file(spec['predictions_path']),
        }
        for label, spec in experiment_specs.items()
    },
}
with summary_path.open('w', encoding='utf-8') as file:
    json.dump(artifact, file, ensure_ascii=False, indent=2)
with per_image_path.open('w', encoding='utf-8', newline='') as file:
    writer = csv.DictWriter(
        file,
        fieldnames=[
            'experiment', 'image_id', 'prediction', 'CIDEr', 'BLEU-4', 'CLIPScore'
        ],
    )
    writer.writeheader()
    writer.writerows(per_image_rows)

print('\nEvaluation ranking')
print(f"{'Rank':>4}  {'Experiment':<28} {'CIDEr':>10} {'BLEU-4':>10} {'CLIPScore':>12}")
for row in summary_rows:
    print(
        f"{row['rank']:>4}  {row['experiment']:<28} "
        f"{row['CIDEr']:>10.4f} {row['BLEU-4']:>10.4f} "
        f"{row['CLIPScore']:>12.4f}"
    )
print(f'\nSaved summary: {summary_path}')
print(f'Saved per-image scores: {per_image_path}')

## 6. Quy tắc tuning

Mỗi cấu hình inference dùng một `RUN_TAG` riêng. Chọn cấu hình bằng CIDEr trên validation; nếu gần nhau, dùng BLEU-4. Không chọn bằng CLIPScore và không tiếp tục tuning sau khi mở kết quả test.